# **DATA EXPLORATION AND PREPROCESSING**

---

## **1. Import Libraries**

In [195]:
import pandas as pd
import numpy as np
import sys
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.compose import ColumnTransformer

## **2. Load Raw Data**

Trong giai đoạn đầu của quá trình khám phá dữ liệu (EDA), chúng em tiến hành tải dữ liệu thô (raw data) từ tệp CSV được lưu trữ trong cấu trúc thư mục dự án (`data/raw`).

In [196]:
energy_path = "../../data/raw/Energy_Use.csv"
df_energy = pd.read_csv(energy_path)

Dữ liệu được đọc từ tệp `Energy_Use.csv` và được khởi tạo dưới dạng cấu trúc bảng lưu vào biến DataFrame `df_energy`.

## **3. Basic Dataset Overview**

### **3.1 Dataset Dimensions**

Bước đầu tiên để nắm bắt quy mô của dữ liệu là kiểm tra chiều không gian (số lượng quan sát và số lượng đặc trưng).

In [197]:
num_rows_energy, num_cols_energy = df_energy.shape
print(f"Number of rows: {num_rows_energy}")
print(f"Number of columns: {num_cols_energy}")

Number of rows: 19735
Number of columns: 29


Bộ dữ liệu **Appliances Energy Prediction** bao gồm *19.735 quan sát* và *29 biến đặc trưng*. Cụ thể, không gian đặc trưng bao gồm 27 biến liên tục (đo lường nhiệt độ, độ ẩm trong nhà/ngoài trời, và các thông số thời tiết khác), 1 biến thời gian (`date`) và 1 biến mục tiêu (`Appliances` - lượng điện năng tiêu thụ). Quy mô và sự đa dạng của các biến số này cung cấp một nền tảng vững chắc để thực hiện các phân tích tương quan đa biến và xây dựng các mô hình Hồi quy (Regression) dự báo mức tiêu thụ năng lượng.

### **3.2 Observational Unit**

Đơn vị quan sát cốt lõi của tập dữ liệu này được xác định như sau:

> **Một bản ghi tương ứng với một chu kỳ đo lường tổng hợp kéo dài 10 phút tại một ngôi nhà duy nhất.**

**Đặc điểm cấu trúc dữ liệu:**
Khác với cấu trúc panel-like data ở nhiều địa điểm, bộ dữ liệu này thuần túy mang cấu trúc *Time-series* tần suất cao. Các tín hiệu từ mạng lưới cảm biến không dây ZigBee trong nhà (thu thập mỗi 3.3 phút) và dữ liệu thời tiết thực tế từ trạm khí tượng đã được đồng bộ hóa và tính trung bình theo chu kỳ 10 phút liên tục trong suốt 4.5 tháng.

### **3.3 Initial Data Glimpse**

Để có cái nhìn ban đầu về cấu trúc và đặc điểm của dataset, chúng em hiển thị một số dòng đầu và cuối để kiểm tra định dạng dữ liệu và mức độ nhất quán:

In [198]:
# Xem 5 dòng đầu
df_energy.head()

,date,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,...,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2
0,11-01-2016 17:00,60,30,19.89,47.596667,19.2,44.790000,19.79,44.730000,19.000000,...,17.033333,45.53,6.60,733.5,92.0,7.000000,63.000000,5.3,13.275433,13.275433
1,11-01-2016 17:10,60,30,19.89,46.693333,19.2,44.722500,19.79,44.790000,19.000000,...,17.066667,45.56,6.48,733.6,92.0,6.666667,59.166667,5.2,18.606195,18.606195
2,11-01-2016 17:20,50,30,19.89,46.300000,19.2,44.626667,19.79,44.933333,18.926667,...,17.000000,45.50,6.37,733.7,92.0,6.333333,55.333333,5.1,28.642668,28.642668
3,11-01-2016 17:30,50,40,19.89,46.066667,19.2,44.590000,19.79,45.000000,18.890000,...,17.000000,45.40,6.25,733.8,92.0,6.000000,51.500000,5.0,45.410390,45.410390
4,11-01-2016 17:40,60,40,19.89,46.333333,19.2,44.530000,19.79,45.000000,18.890000,...,17.000000,45.40,6.13,733.9,92.0,5.666667,47.666667,4.9,10.084097,10.084097


In [199]:
# Xem 5 dòng cuối
df_energy.tail()

,date,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,...,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2
19730,27-05-2016 17:20,100,0,25.566667,46.560000,25.890000,42.025714,27.200000,41.163333,24.7,...,23.2,46.7900,22.7,755.2,55.666667,3.333333,23.666667,13.3,43.096812,43.096812
19731,27-05-2016 17:30,90,0,25.500000,46.500000,25.754000,42.080000,27.133333,41.223333,24.7,...,23.2,46.7900,22.6,755.2,56.000000,3.500000,24.500000,13.3,49.282940,49.282940
19732,27-05-2016 17:40,270,10,25.500000,46.596667,25.628571,42.768571,27.050000,41.690000,24.7,...,23.2,46.7900,22.5,755.2,56.333333,3.666667,25.333333,13.3,29.199117,29.199117
19733,27-05-2016 17:50,420,10,25.500000,46.990000,25.414000,43.036000,26.890000,41.290000,24.7,...,23.2,46.8175,22.3,755.2,56.666667,3.833333,26.166667,13.2,6.322784,6.322784
19734,27-05-2016 18:00,430,10,25.500000,46.600000,25.264286,42.971429,26.823333,41.156667,24.7,...,23.2,46.8450,22.2,755.2,57.000000,4.000000,27.000000,13.2,34.118851,34.118851


**Summary of Column Types**

In [200]:
df_energy.dtypes.value_counts()

float64    26
int64       2
object      1
Name: count, dtype: int64

Kết quả phân tích kiểu dữ liệu cho thấy tập dữ liệu `df_energy` cực kỳ lý tưởng cho các bài toán Học máy định lượng. Có tới 28/29 thuộc tính mang kiểu dữ liệu số (`float64` và `int64`), đại diện cho các thông số đo lường vật lý liên tục từ cảm biến (nhiệt độ, độ ẩm, áp suất, tốc độ gió) và mức tiêu thụ điện năng.

Biến duy nhất có kiểu `object` (chuỗi ký tự) là cột `date`. Từ góc độ tiền xử lý, định dạng chuỗi của cột này chưa thể sử dụng trực tiếp cho các phép toán chuỗi thời gian, do đó yêu cầu bắt buộc là phải ép kiểu (type-casting) cột này về định dạng `datetime` chuẩn của Pandas trước khi tiến hành phân tích sâu hơn.

### **3.4 Temporal Coverage & Spatial Distribution**

In [201]:
df_energy['date'] = pd.to_datetime(df_energy['date'], format='%d-%m-%Y %H:%M')
print(df_energy['date'].min(), "đến", df_energy['date'].max())

2016-01-11 17:00:00 đến 2016-05-27 18:00:00


**Temporal Coverage:** Dữ liệu trải dài liên tục từ chiều ngày `11/01/2016` đến chiều ngày `27/05/2016` (khoảng 4.5 tháng). Tần suất lấy mẫu dày đặc (mỗi 10 phút/lần) tạo ra một chuỗi thời gian có độ phân giải cao (high-resolution time-series), rất phù hợp để bắt được các chu kỳ sử dụng điện năng ngắn hạn (theo giờ trong ngày) và trung hạn (theo ngày trong tuần).

**Spatial Distribution:** Khác với bộ dữ liệu khí tượng diện rộng, toàn bộ tập dữ liệu này được thu thập tại một địa điểm duy nhất (một ngôi nhà tiết kiệm năng lượng tại Bỉ). Sự cố định về mặt không gian này loại bỏ được nhiễu loạn do sai biệt địa lý, giúp mô hình tập trung hoàn toàn vào việc học các quy luật nhiệt động lực học trong nhà và tác động của thời tiết bên ngoài lên hành vi tiêu thụ điện của gia đình đó.

## **4. Data Semantics**

Phần này phân tích ngữ nghĩa của dữ liệu nhằm xác định ý nghĩa thực tế của các quan sát và các biến số, từ đó vạch ra chiến lược trích xuất đặc trưng phù hợp cho mô hình Hồi quy.

### **4.1 The meaning of each row**

Mỗi dòng trong bộ dữ liệu **Appliances Energy Prediction** đại diện cho một *bản ghi trạng thái tổng hợp trong khoảng thời gian 10 phút* của ngôi nhà.

Trong 10 phút này, hệ thống ghi nhận:

1. Mức điện năng tiêu thụ thực tế của thiết bị gia dụng và đèn chiếu sáng (qua đồng hồ m-bus).
2. Giá trị trung bình của các thông số môi trường trong nhà (nhiệt độ, độ ẩm) được truyền về từ mạng lưới cảm biến không dây ZigBee.
3. Điều kiện thời tiết ngoài trời tại cùng thời điểm đó được nội suy từ trạm khí tượng gần nhất.

**Tính chất cốt lõi:** Dữ liệu mang cấu trúc *Chuỗi thời gian đa biến (Multivariate Time-series Data)*.

**Lưu ý:** Quán tính nhiệt của ngôi nhà khiến nhiệt độ ở phút thứ $t$ phụ thuộc rất mạnh vào phút thứ $t-1$. Tương tự, hành vi bật/tắt thiết bị của con người cũng có tính kế thừa thời gian. Do đó, trong bước mô hình hóa hoặc chia tập Train/Test, ta tuyệt đối *không được dùng random shuffle* dữ liệu mà phải duy trì trật tự thời gian (ví dụ: dùng TimeSeriesSplit) để tránh hiện tượng rò rỉ dữ liệu.

### **4.2 The meaning of each column**

In [202]:
df_energy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19735 entries, 0 to 19734
Data columns (total 29 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         19735 non-null  datetime64[ns]
 1   Appliances   19735 non-null  int64         
 2   lights       19735 non-null  int64         
 3   T1           19735 non-null  float64       
 4   RH_1         19735 non-null  float64       
 5   T2           19735 non-null  float64       
 6   RH_2         19735 non-null  float64       
 7   T3           19735 non-null  float64       
 8   RH_3         19735 non-null  float64       
 9   T4           19735 non-null  float64       
 10  RH_4         19735 non-null  float64       
 11  T5           19735 non-null  float64       
 12  RH_5         19735 non-null  float64       
 13  T6           19735 non-null  float64       
 14  RH_6         19735 non-null  float64       
 15  T7           19735 non-null  float64       
 16  RH_7

Không gian đặc trưng gồm 29 biến được phân loại thành 5 nhóm ngữ nghĩa chính dựa trên nguồn gốc sinh ra dữ liệu:

**1. Biến thời gian (Temporal Feature):**
* `date`: Thời điểm ghi nhận (YYYY-MM-DD HH:MM:SS). Từ biến này, ta có thể trích xuất thêm các đặc trưng chu kỳ ẩn như giờ trong ngày, ngày trong tuần, tháng trong năm.


**2. Biến mục tiêu và năng lượng (Target & Energy Features):**
* `Appliances`: *(Biến mục tiêu - Target)* Lượng điện năng tiêu thụ của các thiết bị gia dụng chính (đơn vị: Wh). Đây là đại lượng liên tục cần dự báo.
* `lights`: Lượng điện tiêu thụ của các thiết bị chiếu sáng (Wh). Mặc dù là năng lượng, biến này có thể được dùng như một đặc trưng dự báo vì việc bật đèn thường là chỉ báo trực tiếp cho sự hiện diện của con người trong nhà, từ đó kéo theo việc sử dụng các thiết bị khác.

**3. Nhóm biến Môi trường trong nhà (Indoor Thermodynamic Features):**
Bao gồm 18 biến đo lường từ 9 khu vực khác nhau trong nhà. Sự phân chia chi tiết này cho phép đánh giá mức độ truyền nhiệt giữa các phòng:
* *Khu vực sinh hoạt chính:* `T1`/`RH_1` (Bếp), `T2`/`RH_2` (Phòng khách).
* *Khu vực phụ trợ/Sinh hoạt riêng:* `T3`/`RH_3` (Phòng giặt), `T4`/`RH_4` (Phòng làm việc), `T5`/`RH_5` (Phòng tắm), `T7`/`RH_7` (Phòng ủi đồ), `T8`/`RH_8` (Phòng ngủ 2), `T9`/`RH_9` (Phòng ngủ chính).
* *(Dự báo nguy cơ)*: Các biến nhiệt độ trong nhà khả năng cao sẽ xảy ra hiện tượng đa cộng tuyến (Multicollinearity) vì nhiệt độ các phòng thường tăng giảm cùng nhau.

**4. Nhóm biến Thời tiết ngoài trời (Outdoor Meteorological Features):**
Bao gồm các yếu tố tác động trực tiếp đến lớp vỏ công trình và nhu cầu làm mát/sưởi ấm:
* `T6`, `RH_6`: Nhiệt độ và độ ẩm ngay bên ngoài mặt Bắc của ngôi nhà.
* Dữ liệu từ trạm khí tượng: `T_out` (Nhiệt độ môi trường), `RH_out` (Độ ẩm), `Press_mm_hg` (Áp suất), `Windspeed` (Tốc độ gió), `Visibility` (Tầm nhìn), `Tdewpoint` (Nhiệt độ đọng sương).

**5. Nhóm biến Kiểm định (Random Variables):**
* `rv1`, `rv2`: Hai biến số ngẫu nhiên hoàn toàn không có ý nghĩa vật lý. Chúng được các nhà nghiên cứu cố tình đưa vào tập dữ liệu để làm mốc tham chiếu khi đánh giá tầm quan trọng của các đặc trưng. Bất kỳ đặc trưng vật lý nào có độ quan trọng thấp hơn hoặc bằng hai biến ngẫu nhiên này đều có thể bị loại bỏ để giảm chiều dữ liệu.

### **4.3 Column Data Types and Compatibility for Analysis**

In [203]:
dtype_summary = df_energy.dtypes.value_counts().reset_index()
dtype_summary.columns = ['Data Type', 'Count']
dtype_summary['Example Features'] = dtype_summary['Data Type'].apply(
    lambda x: list(df_energy.select_dtypes(include=x).columns[:3])
)

print("Data Type Distribution:")
display(dtype_summary)
print("\nDetailed Feature Types:")
numerical_feats = df_energy.select_dtypes(include=['float64', 'int64']).columns
categorical_feats = df_energy.select_dtypes(include=['object']).columns
print(f" - Numerical Features ({len(numerical_feats)}): {list(numerical_feats)}")
print(f" - Categorical Features ({len(categorical_feats)}): {list(categorical_feats)}")

Data Type Distribution:


,Data Type,Count,Example Features
0,float64,26,"[T1, RH_1, T2]"
1,int64,2,"[Appliances, lights]"
2,datetime64[ns],1,[date]



Detailed Feature Types:
 - Numerical Features (28): ['Appliances', 'lights', 'T1', 'RH_1', 'T2', 'RH_2', 'T3', 'RH_3', 'T4', 'RH_4', 'T5', 'RH_5', 'T6', 'RH_6', 'T7', 'RH_7', 'T8', 'RH_8', 'T9', 'RH_9', 'T_out', 'Press_mm_hg', 'RH_out', 'Windspeed', 'Visibility', 'Tdewpoint', 'rv1', 'rv2']
 - Categorical Features (0): []


Kết quả phân tích cho thấy cấu trúc của tập dữ liệu `df_energy` cực kỳ lý tưởng và tinh gọn cho bài toán Hồi quy (Regression). Không giống như các tập dữ liệu khí tượng diện rộng thường chứa nhiều thông tin phân loại, toàn bộ các đặc trưng dự báo ở đây đều đã ở dạng định lượng.

**Nhóm dữ liệu định lượng (Numerical - float64 & int64):** Chiếm thế áp đảo với 28 biến. Nhóm này trực tiếp biểu diễn các đại lượng vật lý liên tục như điện năng (Wh), nhiệt độ (°C), độ ẩm (%), áp suất (mm Hg) và tốc độ gió (m/s).
* *Định hướng xử lý:* Nhờ bản chất là dữ liệu số, các thuật toán Học máy có thể tính toán trực tiếp mà không cần qua bước mã hóa (Encoding). Tuy nhiên, do các biến này có thang đo (scale) hoàn toàn khác nhau (ví dụ: Áp suất ~700, Nhiệt độ ~20), một bước tiền xử lý *bắt buộc* là phải thực hiện *Chuẩn hóa dữ liệu (Standardization/Min-Max Scaling)* để đảm bảo các mô hình nhạy cảm với khoảng cách (như SVM, KNN, hoặc Neural Networks) có thể hội tụ và hoạt động chính xác.

**Nhóm dữ liệu phân loại (Categorical - object):** Hoàn toàn vắng mặt (0 biến). Việc không có biến chuỗi/phân loại giúp loại bỏ hoàn toàn rủi ro bùng nổ số chiều dữ liệu (Curse of Dimensionality) thường gặp khi phải áp dụng kỹ thuật One-Hot Encoding.

**Nhóm dữ liệu thời gian (Datetime - datetime64[ns]):** Biến `date` duy nhất đã được ép kiểu thành công ở bước trước.
* *Định hướng xử lý:* Không thể đưa trực tiếp chuỗi thời gian nguyên bản vào hầu hết các mô hình Hồi quy. Cần áp dụng kỹ thuật *Feature Engineering* để phân rã cột `date` này thành các đặc trưng phái sinh mang ý nghĩa chu kỳ sinh hoạt của con người, cụ thể như: `Hour_of_day` (giờ trong ngày), `Day_of_week` (ngày trong tuần), hoặc các cờ nhị phân `is_weekend` (là ngày cuối tuần) và `is_business_hour` (trong giờ hành chính).

## **5. Descriptive Statistics**

 Thống kê mô tả: mean, std, min, max, tứ phân vị.

## **6. Target Variable Distribution**

Phân bố của biến mục tiêu (histogram, boxplot).

*(Bao gồm: Vẽ Histogram để xem hình dáng phân bố, Boxplot để xem phổ giá trị của biến `Appliances`. Đánh giá xem dữ liệu có bị lệch - skewness hay không để định hướng thuật toán).*

## **7. Correlation & Bivariate Analysis**

*(Bao gồm: Vẽ Heatmap cho Ma trận tương quan (Correlation Matrix) để tìm ra các biến môi trường/thời tiết có ảnh hưởng mạnh nhất đến biến mục tiêu. Vẽ Scatter plot cho các cặp biến quan trọng nhất để xem mối quan hệ là tuyến tính hay phi tuyến).*

### **7.1. Preliminary Patterns: Correlation Matrix**

Trước tiên, nhóm tiến hành tính toán ma trận tương quan cho các biến số để xem xét mức độ liên kết tuyến tính giữa chúng.

In [204]:
# correlation_matrix_heatmap(df_weather)

Ma trận tương quan Pearson này giúp chúng ta định lượng mối quan hệ tuyến tính giữa các biến số. Từ Heatmap, nhóm chia các biến thành các "cụm thông tin" (information clusters) có tác động qua lại mạnh mẽ.

#### **7.1.1. Cụm Đa cộng tuyến cực mạnh (High Multicollinearity)**
Đây là các nhóm biến chứa thông tin gần như trùng lặp, cần lưu ý khi xây dựng các mô hình nhạy cảm với đa cộng tuyến như Logistic Regression.
* Nhóm Nhiệt độ (`MinTemp`, `MaxTemp`, `Temp9am`, `Temp3pm`):
  * Các biến này có hệ số tương quan thuận từ 0.70 đến 0.98.
  * Đáng chú ý nằm ở 2 biến `MaxTemp` và `Temp3pm` có tương quan gần như tuyệt đối ($\approx 0.98$). Điều này hợp lý vì nhiệt độ cao nhất trong ngày thường rơi vào khoảng xế chiều.
  * Nhóm hướng đến việc xử lí bằng cách trong các bước Feature Selection, chỉ cần giữ lại `MaxTemp` hoặc tính toán biến `TempRange = MaxTemp - MinTemp` để nắm bắt sự biến động nhiệt độ thay vì dùng cả 4 biến.
* Nhóm Áp suất (`Pressure9am`, `Pressure3pm`):
  * Có hệ số tương quan thuận cực mạnh (0.96). Điều này cho thấy áp suất khí quyển thường thay đổi rất chậm trong ngày.
  * Hướng xử lí: Có thể sử dụng giá trị trung bình hoặc chỉ giữ lại một biến vì chúng cung cấp thông tin dự báo tương đương nhau.

#### **7.1.2. Mối quan hệ nghịch biến giữa Sunshine, Cloud và Humidity**
Đây là nhóm biến chứa thông tin quan trọng để dự báo khả năng có mưa.
* **Sunshine với Cloud3pm/Cloud9am**:
  * Hệ số tương quan nghịch mạnh (khoảng -0.67 đến -0.71).
  * Lý do là vì lượng mây che phủ càng lớn thì số giờ nắng càng thấp. Mối quan hệ này rất chặt chẽ và nhất quán, cho thấy độ tin cậy của dữ liệu thu thập.
* **Sunshine với Humidity3pm**:
  * Hệ số tương quan nghịch khoảng -0.62.
  * Nhận xét: Khi độ ẩm buổi chiều cao, thường đi kèm với việc hình thành mây, dẫn đến giảm số giờ nắng. Đây là dấu hiệu tiền đề của các cơn mưa vào ngày hôm sau.

#### **7.1.3. Sự khác biệt giữa các mốc thời gian (9am và 3pm)**
* **Humidity**: `Humidity3pm` thường có tương quan với khả năng mưa cao hơn so với `Humidity9am`.
* **Wind Speed**: `WindGustSpeed` (tốc độ gió giật cao nhất) có tương quan thuận với cả `WindSpeed9am` và `WindSpeed3pm`, nhưng thường mạnh hơn ở mốc 3pm. Điều này gợi ý rằng các biến số đo lường vào buổi chiều mang nhiều tín hiệu dự báo cho ngày mai hơn là các biến buổi sáng.

#### **7.1.4. Những mối tương quan yếu gây bất ngờ**
* **Rainfall với các biến khác**: Lượng mưa của ngày hôm nay (`Rainfall`) có tương quan khá thấp ($< 0.3$) với các biến số khác như Áp suất hay Nhiệt độ. Điều này có thể lí giải bởi trường hợp mưa là một hiện tượng phi tuyến tính phức tạp. Việc hôm nay mưa bao nhiêu không phụ thuộc tuyến tính đơn giản vào việc hôm nay nóng bao nhiêu. Điều này cho thấy việc sử dụng các mô hình học máy phi tuyến (như Random Forest hoặc XGBoost) sẽ hiệu quả hơn các mô hình tuyến tính đơn giản.
* **WindGustSpeed với Pressure**: Có hệ số tương quan nghịch nhẹ, cho thấy rằng khi áp suất giảm đột ngột (rãnh thấp), tốc độ gió thường có xu hướng tăng lên.

## **8. Outlier Detection** 

*(Bao gồm: Sử dụng phương pháp IQR hoặc Z-score trên các biến quan trọng. Giải thích nguyên nhân xuất hiện ngoại lai (do lỗi cảm biến hay do thực tế sử dụng) và đề xuất hướng xử lý: giữ nguyên, cắt xén (clipping), hoặc loại bỏ).*

## **9. Data Preprocessing & Feature Engineering**

### **9.1 Missing Data**

Phân tích dữ liệu thiếu là bước then chốt để đánh giá mức độ hoàn thiện và độ tin cậy của bộ dữ liệu. Đối với dữ liệu chuỗi thời gian, mục tiêu của phần này là rà soát toàn diện nhằm xác định xem có bất kỳ sự đứt gãy thông tin nào từ các thiết bị đo lường (như đồng hồ điện, mạng lưới cảm biến) hay không. Việc xác nhận một bộ dữ liệu đầy đủ sẽ giúp chúng ta đảm bảo không làm sai lệch các quy luật vật lý và hành vi tiêu thụ năng lượng nguyên bản của ngôi nhà.

#### **Missing Summary Table**

Chúng ta bắt đầu bằng cái nhìn tổng quan định lượng về tỷ lệ thiếu của từng biến số.

In [205]:
# Tạo bảng thống kê dữ liệu thiếu
missing_counts = df_energy.isnull().sum()
total_cells = df_energy.size
total_missing = missing_counts.sum()

missing_df = pd.DataFrame({
    'Feature': df_energy.columns,
    'Missing Count': missing_counts.values,
    'Missing %': (missing_counts.values / len(df_energy)) * 100
}).sort_values(by='Missing Count', ascending=False)

cols_with_missing = (missing_df["Missing Count"] > 0).sum()
overall_missing_rate = (total_missing / total_cells) * 100

print("Dataset Missing Overview:")
print(f"- Total Missing Values    : {total_missing:,.0f}")
print(f"- Overall Missing Rate    : {overall_missing_rate:.2f}%")
print(f"- Columns With Missing    : {cols_with_missing} / {len(df_energy.columns)}\n")

print("Detailed Missingness by Feature (Descending):")

# Hiển thị bảng định dạng với thanh màu
display(
    missing_df.style
        .format({"Missing %": "{:.2f}%"})
        .bar(subset=["Missing %"], color="#FF6B6B", vmin=0, vmax=100)
        .background_gradient(subset=["Missing Count"], cmap="Reds")
)


Dataset Missing Overview:
- Total Missing Values    : 0
- Overall Missing Rate    : 0.00%
- Columns With Missing    : 0 / 29

Detailed Missingness by Feature (Descending):


,Feature,Missing Count,Missing %
0,date,0,0.00%
1,Appliances,0,0.00%
2,lights,0,0.00%
3,T1,0,0.00%
4,RH_1,0,0.00%
5,T2,0,0.00%
6,RH_2,0,0.00%
7,T3,0,0.00%
8,RH_3,0,0.00%
9,T4,0,0.00%


#### **Nhận xét & Chiến lược xử lý**

Như kết quả thống kê ở trên, bộ dữ liệu **Appliances Energy Prediction** đạt mức độ hoàn thiện lý tưởng với **0% dữ liệu thiếu** trên toàn bộ 29 không gian đặc trưng. Cả 19.735 quan sát đều có đầy đủ thông tin từ biến thời gian, biến mục tiêu cho đến hệ thống cảm biến môi trường.

**Nguyên nhân dữ liệu sạch:**
Sự trọn vẹn này có được là nhờ tính chất tự động hóa của hệ thống thu thập dữ liệu IoT (mạng lưới cảm biến không dây ZigBee trong nhà và trạm khí tượng tự động). Cứ mỗi 10 phút, hệ thống sẽ tự động đồng bộ và nội suy các thông số mà không phụ thuộc vào sự ghi chép thủ công của con người.

**Chiến lược xử lý:**
* **Bỏ qua bước Imputation:** Do không có giá trị `NaN`, chúng ta hoàn toàn không cần áp dụng các kỹ thuật điền khuyết (như Mean, Median hay MICE).
* **Tiến thẳng đến Feature Engineering:** Dữ liệu đã sẵn sàng để chuyển sang các bước biến đổi đặc trưng và chia tập (Train/Test Split).


### **9.2 Feature Engineering**

Tiếp theo, nhóm sẽ tiến hành trích xuất các đặc trưng mới từ dữ liệu gốc nhằm giúp thuật toán học máy bắt được các chu kỳ sinh hoạt của con người và quy luật truyền nhiệt của ngôi nhà một cách rõ ràng nhất.

In [206]:
# Tạo bản sao để tránh làm thay đổi dữ liệu gốc ban đầu
df_fe = df_energy.copy()

# Chuyển cột date sang định dạng datetime
df_fe['date'] = pd.to_datetime(df_fe['date'])

# Bóc tách các thành phần thời gian
df_fe['Month'] = df_fe['date'].dt.month
df_fe['DayOfWeek'] = df_fe['date'].dt.dayofweek # Thứ 2 = 0, Thứ 3 = 1, ... , Chủ nhật = 6
df_fe['Hour'] = df_fe['date'].dt.hour
df_fe['Minute'] = df_fe['date'].dt.minute

# Tạo cờ cuối tuần (Thứ 7, CN)
df_fe['Is_Weekend'] = df_fe['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)

# Tạo cờ giờ hành chính (8h sáng đến 17h chiều, và không phải cuối tuần)
df_fe['Is_Business_Hour'] = ((df_fe['Hour'] >= 8) & (df_fe['Hour'] < 17) & (df_fe['Is_Weekend'] == 0)).astype(int)

# Tính nhiệt độ trung bình các phòng trong nhà (Từ T1 đến T9, bỏ T6 vì T6 là mặt ngoài Bắc ngôi nhà)
indoor_temp_cols = ['T1', 'T2', 'T3', 'T4', 'T5', 'T7', 'T8', 'T9']
df_fe['T_indoor_avg'] = df_fe[indoor_temp_cols].mean(axis=1)

# Tính độ chênh lệch nhiệt độ trong và ngoài nhà
df_fe['Temp_Diff'] = df_fe['T_indoor_avg'] - df_fe['T_out']

cols_to_show = ['date', 'Month', 'DayOfWeek', 'Hour', 'Minute','Is_Weekend', 'Is_Business_Hour', 'T_indoor_avg', 'Temp_Diff']
print("\n10 dòng dữ liệu ngẫu nhiên của các biến mới tạo")
display(df_fe[cols_to_show].sample(n=10, random_state=42).sort_index())

# Drop cột date sau khi kiểm tra xong
df_fe = df_fe.drop(columns=['date'], errors='ignore')


10 dòng dữ liệu ngẫu nhiên của các biến mới tạo


,date,Month,DayOfWeek,Hour,Minute,Is_Weekend,Is_Business_Hour,T_indoor_avg,Temp_Diff
2754,2016-01-30 20:00:00,1,5,20,0,1,0,19.602639,16.902639
7801,2016-03-05 21:10:00,3,5,21,10,1,0,19.095000,15.925000
8348,2016-03-09 16:20:00,3,2,16,20,0,1,18.696667,12.126667
8875,2016-03-13 08:10:00,3,6,8,10,1,0,19.383333,19.850333
8980,2016-03-14 01:40:00,3,0,1,40,0,0,20.157344,18.387344
9132,2016-03-15 03:00:00,3,1,3,0,0,0,20.357500,20.157500
10239,2016-03-22 19:30:00,3,1,19,30,0,0,21.062917,13.512917
13166,2016-04-12 03:20:00,4,1,3,20,0,0,21.612917,14.012917
14359,2016-04-20 10:10:00,4,2,10,10,0,1,21.359583,12.509583
15847,2016-04-30 18:10:00,4,5,18,10,1,0,21.077083,11.397083


Xuất phát từ đặc thù của mạng lưới cảm biến và tính chu kỳ trong hành vi sinh hoạt, quá trình kỹ nghệ đặc trưng được thiết kế xoay quanh hai nhóm biến số trọng tâm:

**1. Trích xuất đặc trưng thời gian:**
* **Vấn đề:** Biến `date` là chuỗi thời gian liên tục, các thuật toán Hồi quy hay Cây quyết định không thể đọc và tính toán trực tiếp trên định dạng này.
* **Giải pháp & Lý do lựa chọn:** Phân rã `date` thành các đặc trưng rời rạc để mô phỏng chính xác nhịp điệu sinh hoạt của gia đình:
  * `Month` (Tháng trong năm): Giúp thuật toán nhận biết sự thay đổi của thời tiết theo tháng, từ đó dự báo xu hướng tiêu thụ năng lượng thiết bị làm mát/sưởi ấm.
  * `DayOfWeek` (Ngày trong tuần) & `Is_Weekend` (Cuối tuần): Tần suất có mặt ở nhà và nhu cầu nấu nướng, giặt giũ vào ngày nghỉ (Thứ 7, Chủ nhật) thường cao hơn hẳn các ngày đi làm trong tuần.
  * `Hour` (Giờ trong ngày): Là yếu tố quan trọng nhất. Hành vi dùng điện có sự phân hóa rõ rệt theo giờ (ban đêm dùng nhiều thiết bị hơn ban ngày).
  * `Minute` (Phút trong giờ): Do dữ liệu được ghi nhận với tần suất 10 phút/lần, việc bổ sung biến Phút giúp mô hình nắm bắt được chu kì hoạt động của thiết bị (ví dụ: chu kỳ đóng/ngắt tự động của tủ lạnh, máy sưởi) – những biến động ngắn hạn mà đặc trưng Giờ không đủ độ phân giải để phản ánh.
  * `Is_Business_Hour` (Giờ hành chính): Xác định khung giờ làm việc (từ 8h sáng đến 5h chiều) của các ngày trong tuần. Trong khoảng thời gian này, nhà thường trống nên mức tiêu thụ năng lượng sẽ tụt xuống mức nền.
 
**2. Tạo đặc trưng nhiệt động lực học:**
* **Vấn đề:** Dữ liệu cung cấp nhiệt độ của 9 phòng khác nhau, nhưng năng lượng tổng (`Appliances`) lại phục vụ cho cả ngôi nhà.
* **Giải pháp & Lý do lựa chọn:** 
  * `T_indoor_avg` (Nhiệt độ trung bình trong nhà): Gộp nhiệt độ các phòng (trừ mặt ngoài T6) để tạo ra một đại lượng đại diện cho mức nhiệt năng chung của toàn bộ không gian sống.
  * `Temp_Diff` (Độ chênh lệch nhiệt độ): Bằng Nhiệt độ trung bình trong nhà trừ đi Nhiệt độ ngoài trời (`T_out`). Theo nguyên lý nhiệt động lực học, mức độ chênh lệch nhiệt độ giữa trong và ngoài vỏ công trình càng lớn thì hệ thống điều hòa/sưởi ấm càng phải hoạt động với công suất cao để bù đắp.

#### **Cyclical Feature Encoding**

In [207]:
# 1. Mã hóa cho Tháng (Chu kỳ 12)
df_fe['Month_sin'] = np.sin(2 * np.pi * df_fe['Month'] / 12)
df_fe['Month_cos'] = np.cos(2 * np.pi * df_fe['Month'] / 12)

# 2. Mã hóa cho Ngày trong tuần (Chu kỳ 7)
df_fe['DayOfWeek_sin'] = np.sin(2 * np.pi * df_fe['DayOfWeek'] / 7)
df_fe['DayOfWeek_cos'] = np.cos(2 * np.pi * df_fe['DayOfWeek'] / 7)

# 3. Mã hóa cho Giờ (Chu kỳ 24)
df_fe['Hour_sin'] = np.sin(2 * np.pi * df_fe['Hour'] / 24)
df_fe['Hour_cos'] = np.cos(2 * np.pi * df_fe['Hour'] / 24)

# 4. Mã hóa cho Phút (Chu kỳ 60)
df_fe['Minute_sin'] = np.sin(2 * np.pi * df_fe['Minute'] / 60)
df_fe['Minute_cos'] = np.cos(2 * np.pi * df_fe['Minute'] / 60)

# Lấy 10 dòng ngẫu nhiên và sắp xếp theo thời gian để xem kết quả
cols_to_show = [
    'Month', 'DayOfWeek', 'Hour', 'Minute',
    'Month_sin', 'Month_cos', 
    'DayOfWeek_sin', 'DayOfWeek_cos', 
    'Hour_sin', 'Hour_cos', 
    'Minute_sin', 'Minute_cos'
]
display(df_fe[cols_to_show].sample(n=10, random_state=42).sort_index())

# Xóa bỏ các cột số nguyên gốc để tránh đa cộng tuyến
df_fe = df_fe.drop(columns=['Month', 'DayOfWeek', 'Hour', 'Minute'])


,Month,DayOfWeek,Hour,Minute,Month_sin,Month_cos,DayOfWeek_sin,DayOfWeek_cos,Hour_sin,Hour_cos,Minute_sin,Minute_cos
2754,1,5,20,0,0.500000,8.660254e-01,-0.974928,-0.222521,-0.866025,5.000000e-01,0.000000e+00,1.0
7801,3,5,21,10,1.000000,6.123234e-17,-0.974928,-0.222521,-0.707107,7.071068e-01,8.660254e-01,0.5
8348,3,2,16,20,1.000000,6.123234e-17,0.974928,-0.222521,-0.866025,-5.000000e-01,8.660254e-01,-0.5
8875,3,6,8,10,1.000000,6.123234e-17,-0.781831,0.623490,0.866025,-5.000000e-01,8.660254e-01,0.5
8980,3,0,1,40,1.000000,6.123234e-17,0.000000,1.000000,0.258819,9.659258e-01,-8.660254e-01,-0.5
9132,3,1,3,0,1.000000,6.123234e-17,0.781831,0.623490,0.707107,7.071068e-01,0.000000e+00,1.0
10239,3,1,19,30,1.000000,6.123234e-17,0.781831,0.623490,-0.965926,2.588190e-01,5.665539e-16,-1.0
13166,4,1,3,20,0.866025,-5.000000e-01,0.781831,0.623490,0.707107,7.071068e-01,8.660254e-01,-0.5
14359,4,2,10,10,0.866025,-5.000000e-01,0.974928,-0.222521,0.500000,-8.660254e-01,8.660254e-01,0.5
15847,4,5,18,10,0.866025,-5.000000e-01,-0.974928,-0.222521,-1.000000,-1.836970e-16,8.660254e-01,0.5


Các biến thời gian như Giờ trong ngày (`Hour`), Phút trong giờ (`Minute`), Ngày trong tuần (`DayOfWeek`), và Tháng trong năm (`Month`) có tính chất chu kỳ lặp lại. Nếu giữ nguyên dưới dạng số nguyên, mô hình sẽ hiểu sai khoảng cách vật lý. Ví dụ: Sự chênh lệch giữa 23h và 0h sẽ bị mô hình tính toán là 23 đơn vị khoảng cách, trong khi thực tế chúng chỉ cách nhau 1 giờ.

Để giải quyết vấn đề này, nhóm quyết định sử dụng **Mã hóa Sin/Cos**. Bằng cách chiếu các mốc thời gian lên một hệ tọa độ đường tròn lượng giác, điểm cuối chu kỳ sẽ được kết nối mượt mà với điểm đầu chu kỳ. 

### **9.3 Data Splitting Strategy**

* **Tỷ lệ chia:** Dữ liệu được chia thành 3 tập Train / Validation / Test với tỷ lệ tương ứng là **70% / 15% / 15%**.
* **Đặc thù chuỗi thời gian:** Vì dữ liệu phụ thuộc rất lớn vào tính tuần tự của thời gian (nhiệt độ phút này phụ thuộc phút trước), nhóm **không xáo trộn ngẫu nhiên (`shuffle=False`)**. Dữ liệu sẽ được cắt theo đúng trục thời gian tuyến tính để mô hình học từ quá khứ (tháng 1 đến tháng 4) và dự báo cho tương lai (tháng 5), qua đó ngăn chặn rò rỉ dữ liệu tương lai vào tập huấn luyện.

In [208]:
# Tách biến độc lập (X) và biến mục tiêu (y)
X = df_fe.drop(columns=['Appliances'])
y = df_fe['Appliances']

# Chia tập dữ liệu với tỷ lệ 70/15/15 (Không dùng shuffle vì là Time-Series)
# Lần cắt 1: Lấy 85% cho (Train + Val) và 15% cho Test
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, shuffle=False)

# Lần cắt 2: Trong 85% đó, chia lấy phần Validation (chiếm 15%)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=(0.15/0.85), shuffle=False)

print("Kích thước các tập dữ liệu:")
print(f"- Tập Train      : {X_train.shape[0]} dòng ({len(X_train)/len(df_fe)*100:.0f}%)")
print(f"- Tập Validation : {X_val.shape[0]} dòng ({len(X_val)/len(df_fe)*100:.0f}%)")
print(f"- Tập Test       : {X_test.shape[0]} dòng ({len(X_test)/len(df_fe)*100:.0f}%)")

Kích thước các tập dữ liệu:
- Tập Train      : 13813 dòng (70%)
- Tập Validation : 2961 dòng (15%)
- Tập Test       : 2961 dòng (15%)


### **9.4 Feature Scaling**

**Đánh giá phân bố Skewness & Kurtosis để lựa chọn thuật toán Scaling**

Trước khi tiến hành chuẩn hóa dữ liệu, nhóm không lựa chọn thuật toán Scaling một cách cảm tính mà dựa trên cơ sở toán học về hình dáng phân bố của các đặc trưng. Nhóm tiến hành đo lường hai chỉ số thống kê quan trọng:
1. **Độ lệch (Skewness):** Đánh giá tính đối xứng của dữ liệu. Nếu giá trị cách xa 0, dữ liệu bị lệch (trái hoặc phải) rất nặng.
2. **Bề dày đuôi (Kurtosis):** Đánh giá mức độ tập trung của ngoại lai ở phần đuôi. Phân bố chuẩn có Kurtosis bằng 0. Giá trị Kurtosis càng lớn dương chứng tỏ phần đuôi chứa càng nhiều ngoại lai cực đoan.

Mục tiêu của bước này là kiểm tra xem các biến có vi phạm giả định phân phối chuẩn hay không, từ đó có căn cứ loại trừ các thuật toán Scaling không phù hợp.

In [209]:
# 1. Lọc các biến định lượng liên tục 
# bỏ qua các cờ nhị phân
cols_to_exclude = ['Is_Weekend', 'Is_Business_Hour']

# tự động chọn các cột số, không nằm trong danh sách loại trừ, và không có đuôi '_sin' hay '_cos' 
num_features = [col for col in df_fe.columns 
                if col not in cols_to_exclude 
                and not col.endswith('_sin') 
                and not col.endswith('_cos')
                and df_fe[col].dtype in ['float64', 'int64']]

# 2. Tính toán skewness và kurtosis
stats_df = pd.DataFrame({
    'Feature': num_features,
    'Skewness': df_fe[num_features].skew().values,
    'Kurtosis': df_fe[num_features].kurt().values 
}).reset_index(drop=True)

# 3. Phân loại mức độ lệch
def skew_condition(x):
    if abs(x) < 0.5: return "~ chuẩn"
    elif abs(x) < 1: return "lệch vừa"
    else: return "lệch nặng"

stats_df['Đánh giá phân bố'] = stats_df['Skewness'].apply(skew_condition)

# 4. Sắp xếp theo độ lệch giảm dần
stats_df = stats_df.sort_values(by='Skewness', key=abs, ascending=False)

print("Bảng thống kê skewness & kurtosis của từng đặc trưng")
display(stats_df.style.background_gradient(cmap='Oranges', subset=['Skewness', 'Kurtosis']))

Bảng thống kê skewness & kurtosis của từng đặc trưng


,Feature,Skewness,Kurtosis,Đánh giá phân bố
0,Appliances,3.386367,13.667863,lệch nặng
1,lights,2.195155,4.462147,lệch nặng
11,RH_5,1.866820,4.503391,lệch nặng
22,RH_out,-0.922997,0.256859,lệch vừa
4,T2,0.889658,0.933397,lệch vừa
23,Windspeed,0.859982,0.250030,lệch vừa
12,T6,0.597471,0.425549,lệch vừa
10,T5,0.558220,0.112724,lệch vừa
20,T_out,0.534302,0.363283,lệch vừa
7,RH_3,0.467589,-0.583126,~ chuẩn


**Nhận xét và chiến lược xử lý:**

Từ bảng thống kê phân bố trên, nhóm rút ra các kết luận sau:

* **1. Tiêu chí chọn lọc đặc trưng phân tích:** Trước khi đo lường Skewness và Kurtosis, nhóm loại trừ các cờ nhị phân (trạng thái 0/1 như `Is_Weekend`) và các biến chu kỳ lượng giác (`_sin`, `_cos`) vốn ở biên độ [-1, 1]. Lý do là các phép đo hình dáng phân bố chỉ có ý nghĩa thống kê đối với các **biến định lượng vật lý liên tục** (như nhiệt độ, độ ẩm, điện năng). Việc cố gắng nắn chỉnh phân bố trên các biến phân loại hoặc biến trạng thái là phi logic.

* **2. Hệ quả đến việc lựa chọn thuật toán chuẩn hóa:**
  * Việc sử dụng các thuật toán Scale đơn thuần ngay từ đầu là không an toàn. Đặc biệt là **`MinMaxScaler`**, các ngoại lai ở đuôi sẽ đẩy giá trị Max lên quá cao, vô tình ép 99% dữ liệu sinh hoạt bình thường về sát mốc 0. Điều này làm mất phương sai và phá hỏng các hàm phạt của mô hình.
  * Việc dùng **`StandardScaler`** trực tiếp lên dữ liệu gốc cũng sẽ bị khối lượng ngoại lai kéo lệch tham số mean. Tuy nhiên, nếu dữ liệu được xử lý triệt để vấn đề phân bố lệch trước đó, `StandardScaler` lại là sự lựa chọn tối ưu nhất cho các mô hình hồi quy tuyến tính.

* **3. Quyết định cuối cùng:** Dựa trên các minh chứng thống kê này, nhóm quyết định kết hợp **`Power Transformer` (Yeo-Johnson)** và **`StandardScaler`** để chuẩn hóa dữ liệu:
  * **Bước 1 - Nắn hình dáng:** Thuật toán Yeo-Johnson sẽ tự động dò tìm số mũ tối ưu để bẻ cong các phân bố bị lệch về dạng tiệm cận hình chuông đối xứng. Bước này giúp giải quyết triệt để vấn đề Skewness.
  * **Bước 2 - Chuẩn hóa thang đo:** Sau khi dữ liệu đã có hình dáng phân bố chuẩn từ Bước 1, nó thỏa mãn điều kiện lý tưởng của StandardScaler. Thuật toán này sẽ tự tin đưa toàn bộ các đặc trưng về chung một hệ quy chiếu (Trung bình = 0, Độ lệch chuẩn = 1), giúp các thuật toán như Ridge, Lasso hay ElasticNet hội tụ nhanh và dự báo chính xác.

In [210]:
# Tách riêng các nhóm biến
cols_no_scale = [col for col in X_train.columns if col.startswith('Is_') or col.endswith('_sin') or col.endswith('_cos')]
cols_continuous = [col for col in X_train.columns if col not in cols_no_scale]

# Tính toán skewness trên tập train để tự động lọc ra các cột có |skew| > 0.5
skew_series = X_train[cols_continuous].skew()
cols_skewed = skew_series[abs(skew_series) > 0.5].index.tolist()
cols_normal = [col for col in cols_continuous if col not in cols_skewed]

# Thiết lập bộ xử lý riêng biệt
# nhóm lệch nặng: power transformer -> standard scaler
# nhóm còn lại: chỉ standard scaler
preprocessor = ColumnTransformer(
    transformers=[
        ('skewed_pipe', Pipeline([
            ('power', PowerTransformer(method='yeo-johnson', standardize=False)),
            ('scaler', StandardScaler())
        ]), cols_skewed),
        
        ('normal_pipe', StandardScaler(), cols_normal),
        
        ('pass', 'passthrough', cols_no_scale) # giữ nguyên các cột nhị phân/chu kỳ
    ]
)

# Thực thi biến đổi
# ta cần lấy lại danh sách tên cột theo đúng thứ tự mà transformer trả ra
new_col_order = cols_skewed + cols_normal + cols_no_scale

X_train_scaled = pd.DataFrame(preprocessor.fit_transform(X_train), columns=new_col_order, index=X_train.index)
X_val_scaled = pd.DataFrame(preprocessor.transform(X_val), columns=new_col_order, index=X_val.index)
X_test_scaled = pd.DataFrame(preprocessor.transform(X_test), columns=new_col_order, index=X_test.index)


**Save Processed Data**

In [211]:
# Lưu tập Train
df_train_processed = X_train_scaled.copy()
df_train_processed['Appliances'] = y_train.values
df_train_processed.to_csv('../../data/processed/Energy_Use_train.csv', index=False)

# Lưu tập Validation
df_val_processed = X_val_scaled.copy()
df_val_processed['Appliances'] = y_val.values
df_val_processed.to_csv('../../data/processed/Energy_Use_val.csv', index=False)

# Lưu tập Test
df_test_processed = X_test_scaled.copy()
df_test_processed['Appliances'] = y_test.values
df_test_processed.to_csv('../../data/processed/Energy_Use_test.csv', index=False)

# Hiển thị 5 dòng ngẫu nhiên của tập train để kiểm tra
display(X_train_scaled.sample(5, random_state=42))

,lights,T1,T4,RH_5,RH_6,RH_9,RH_out,Windspeed,RH_1,T2,...,Is_Weekend,Is_Business_Hour,Month_sin,Month_cos,DayOfWeek_sin,DayOfWeek_cos,Hour_sin,Hour_cos,Minute_sin,Minute_cos
2270,-0.603703,-1.124590,-0.908766,0.734749,1.212235,1.590167,0.164392,1.715359,1.599666,-0.393325,...,0.0,1.0,0.500000,8.660254e-01,0.974928,-0.222521,0.258819,-0.965926,8.660254e-01,-0.5
5163,-0.603703,-0.078789,-0.060321,-2.501746,-0.236709,-0.239436,-1.538127,-0.128083,-1.126195,-0.106010,...,0.0,1.0,0.866025,5.000000e-01,0.781831,0.623490,-0.258819,-0.965926,5.665539e-16,-1.0
8351,-0.603703,-1.323679,0.300360,-1.268614,-1.202764,-1.533768,-1.477317,0.206392,-1.041803,-1.814184,...,0.0,1.0,1.000000,6.123234e-17,0.974928,-0.222521,-0.866025,-0.500000,-8.660254e-01,0.5
10846,-0.603703,1.662802,0.010621,0.243205,-0.681477,0.146183,-0.468237,1.283069,0.167815,0.975908,...,1.0,0.0,1.000000,6.123234e-17,-0.781831,0.623490,0.000000,1.000000,-8.660254e-01,-0.5
9290,-0.603703,-0.403470,0.172101,-0.454042,-0.079062,-0.659505,0.738932,-0.521772,0.017785,-1.477487,...,0.0,0.0,1.000000,6.123234e-17,0.974928,-0.222521,0.965926,0.258819,8.660254e-01,-0.5


---

# **DATA MODELING AND MODEL EVALUATION**